In [1]:
import json
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import f1_score, classification_report
from sklearn.preprocessing import StandardScaler
import pandas as pd

# ============================================================================
# 1. OPTIMIZED DATASET WITH NORMALIZATION
# ============================================================================
class EmbeddingDataset(Dataset):
    def __init__(self, data, scaler=None, is_test=False, fit_scaler=False):
        self.data = data
        self.is_test = is_test
        
        # Extract features
        features = []
        for record in data:
            feat = record['image_embedding'] + record['text_embedding']
            features.append(feat)
        
        features = np.array(features)
        
        # Normalize features
        if fit_scaler:
            self.scaler = StandardScaler()
            features = self.scaler.fit_transform(features)
        elif scaler is not None:
            self.scaler = scaler
            features = self.scaler.transform(features)
        else:
            self.scaler = None
        
        self.features = features
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        features = torch.FloatTensor(self.features[idx])
        
        if self.is_test:
            return features, self.data[idx]['id']
        else:
            label = torch.LongTensor([self.data[idx]['label']])[0]
            return features, label

# ============================================================================
# 2. IMPROVED MODEL ARCHITECTURE
# ============================================================================
class OptimizedEarlyFusionMLP(nn.Module):
    def __init__(self, input_dim=1024, hidden_dims=[512, 256, 128, 64], 
                 dropout_rate=0.4):
        super().__init__()
        
        layers = []
        prev_dim = input_dim
        
        for i, hidden_dim in enumerate(hidden_dims):
            layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout_rate if i < len(hidden_dims) - 1 else dropout_rate * 0.5)
            ])
            prev_dim = hidden_dim
        
        # Output layer
        layers.append(nn.Linear(prev_dim, 2))
        self.model = nn.Sequential(*layers)
        
        # Initialize weights
        self._init_weights()
    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
    
    def forward(self, x):
        return self.model(x)

# ============================================================================
# 3. TRAINING FUNCTION WITH OPTIMIZATIONS
# ============================================================================
def train_model(train_loader, val_loader, device, config):
    """
    Train model with various optimizations
    
    config: dict with hyperparameters
    """
    model = OptimizedEarlyFusionMLP(
        hidden_dims=config['hidden_dims'],
        dropout_rate=config['dropout_rate']
    ).to(device)
    
    # Loss function with class weights (if imbalanced)
    criterion = nn.CrossEntropyLoss()
    
    # Optimizer with weight decay (L2 regularization)
    optimizer = optim.AdamW(
        model.parameters(), 
        lr=config['learning_rate'],
        weight_decay=config['weight_decay']
    )
    
    # Learning rate scheduler
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, 
        mode='max',  # maximize F1
        factor=0.5, 
        patience=5,
        min_lr=1e-6
    )
    
    # Alternative: CosineAnnealingLR
    # scheduler = optim.lr_scheduler.CosineAnnealingLR(
    #     optimizer, T_max=config['num_epochs'], eta_min=1e-6
    # )
    
    best_f1 = 0
    patience_counter = 0
    train_losses = []
    val_f1_scores = []
    
    for epoch in range(config['num_epochs']):
        # Training phase
        model.train()
        train_loss = 0
        train_preds = []
        train_labels = []
        
        for features, labels in train_loader:
            features, labels = features.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(features)
            loss = criterion(outputs, labels)
            loss.backward()
            
            # Gradient clipping to prevent exploding gradients
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            
            train_loss += loss.item()
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            train_preds.extend(preds)
            train_labels.extend(labels.cpu().numpy())
        
        avg_train_loss = train_loss / len(train_loader)
        train_f1 = f1_score(train_labels, train_preds, average='macro')
        
        # Validation phase
        model.eval()
        val_preds = []
        val_labels = []
        val_loss = 0
        
        with torch.no_grad():
            for features, labels in val_loader:
                features, labels = features.to(device), labels.to(device)
                outputs = model(features)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                
                preds = torch.argmax(outputs, dim=1).cpu().numpy()
                val_preds.extend(preds)
                val_labels.extend(labels.cpu().numpy())
        
        avg_val_loss = val_loss / len(val_loader)
        val_f1 = f1_score(val_labels, val_preds, average='macro')
        
        train_losses.append(avg_train_loss)
        val_f1_scores.append(val_f1)
        
        # Get current learning rate before scheduler step
        current_lr = optimizer.param_groups[0]['lr']
        
        # Update learning rate
        old_lr = current_lr
        scheduler.step(val_f1)
        new_lr = optimizer.param_groups[0]['lr']
        
        # Print progress
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"Epoch {epoch+1}/{config['num_epochs']}")
            print(f"  Train Loss: {avg_train_loss:.4f}, Train F1: {train_f1:.4f}")
            print(f"  Val Loss: {avg_val_loss:.4f}, Val F1: {val_f1:.4f}")
            print(f"  LR: {new_lr:.6f}")
            if new_lr < old_lr:
                print(f"  → Learning rate reduced from {old_lr:.6f} to {new_lr:.6f}")
        
        # Save best model
        if val_f1 > best_f1:
            best_f1 = val_f1
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_f1': best_f1,
                'config': config
            }, 'best_model.pth')
            patience_counter = 0
            print(f"  ✓ New best F1: {best_f1:.4f}")
        else:
            patience_counter += 1
        
        # Early stopping
        if patience_counter >= config['early_stopping_patience']:
            print(f"\nEarly stopping triggered after {epoch+1} epochs")
            break
    
    print(f"\nTraining completed. Best Val F1: {best_f1:.4f}")
    
    # Print final classification report
    print("\nFinal Validation Classification Report:")
    print(classification_report(val_labels, val_preds, 
                                target_names=['Not Important', 'Important']))
    
    return model, best_f1, train_losses, val_f1_scores

# ============================================================================
# 4. HYPERPARAMETER SEARCH
# ============================================================================
def hyperparameter_search(train_data, device):
    """
    Optimized grid search focused on the best performing region
    """
    param_grid = [
        # ===== BASELINE: Current Best =====
        {
            'hidden_dims': [768, 384, 192],
            'dropout_rate': 0.55,
            'learning_rate': 0.001,
            'weight_decay': 1e-5,
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 20
        },
        
        # ===== ULTRA-FINE DROPOUT TUNING (Around 0.55) =====
        {
            'hidden_dims': [768, 384, 192],
            'dropout_rate': 0.54,  # Just below
            'learning_rate': 0.001,
            'weight_decay': 1e-5,
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 20
        },
        {
            'hidden_dims': [768, 384, 192],
            'dropout_rate': 0.56,  # Just above
            'learning_rate': 0.001,
            'weight_decay': 1e-5,
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 20
        },
        {
            'hidden_dims': [768, 384, 192],
            'dropout_rate': 0.535,  # Between 0.525 and 0.55
            'learning_rate': 0.001,
            'weight_decay': 1e-5,
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 20
        },
        {
            'hidden_dims': [768, 384, 192],
            'dropout_rate': 0.545,  # Between 0.54 and 0.55
            'learning_rate': 0.001,
            'weight_decay': 1e-5,
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 20
        },
        
        # ===== ARCHITECTURE MICRO-VARIATIONS (Fixed dropout=0.55) =====
        # Slightly different first layer sizes
        {
            'hidden_dims': [784, 392, 196],  # 1% larger
            'dropout_rate': 0.55,
            'learning_rate': 0.001,
            'weight_decay': 1e-5,
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 20
        },
        {
            'hidden_dims': [752, 376, 188],  # 2% smaller
            'dropout_rate': 0.55,
            'learning_rate': 0.001,
            'weight_decay': 1e-5,
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 20
        },
        # Add one more layer with same pattern
        {
            'hidden_dims': [768, 384, 192, 96],
            'dropout_rate': 0.55,
            'learning_rate': 0.001,
            'weight_decay': 1e-5,
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 20
        },
        # Try 5-layer with adjusted dropout
        {
            'hidden_dims': [768, 512, 384, 256, 192],
            'dropout_rate': 0.52,  # Slightly lower for deeper
            'learning_rate': 0.001,
            'weight_decay': 1e-5,
            'batch_size': 32,
            'num_epochs': 130,
            'early_stopping_patience': 22
        },
        
        # ===== LEARNING RATE MICRO-TUNING =====
        {
            'hidden_dims': [768, 384, 192],
            'dropout_rate': 0.55,
            'learning_rate': 0.0009,  # 10% lower
            'weight_decay': 1e-5,
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 20
        },
        {
            'hidden_dims': [768, 384, 192],
            'dropout_rate': 0.55,
            'learning_rate': 0.0011,  # 10% higher
            'weight_decay': 1e-5,
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 20
        },
        {
            'hidden_dims': [768, 384, 192],
            'dropout_rate': 0.55,
            'learning_rate': 0.00095,  # 5% lower
            'weight_decay': 1e-5,
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 20
        },
        {
            'hidden_dims': [768, 384, 192],
            'dropout_rate': 0.55,
            'learning_rate': 0.00105,  # 5% higher
            'weight_decay': 1e-5,
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 20
        },
        
        # ===== WEIGHT DECAY MICRO-TUNING =====
        {
            'hidden_dims': [768, 384, 192],
            'dropout_rate': 0.55,
            'learning_rate': 0.001,
            'weight_decay': 1.5e-5,  # 50% higher
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 20
        },
        {
            'hidden_dims': [768, 384, 192],
            'dropout_rate': 0.55,
            'learning_rate': 0.001,
            'weight_decay': 7.5e-6,  # 25% lower
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 20
        },
        {
            'hidden_dims': [768, 384, 192],
            'dropout_rate': 0.55,
            'learning_rate': 0.001,
            'weight_decay': 1.2e-5,  # 20% higher
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 20
        },
        
        # ===== BATCH SIZE OPTIMIZATION =====
        {
            'hidden_dims': [768, 384, 192],
            'dropout_rate': 0.55,
            'learning_rate': 0.001,
            'weight_decay': 1e-5,
            'batch_size': 48,  # Middle ground
            'num_epochs': 120,
            'early_stopping_patience': 20
        },
        {
            'hidden_dims': [768, 384, 192],
            'dropout_rate': 0.55,
            'learning_rate': 0.001,
            'weight_decay': 1e-5,
            'batch_size': 24,  # Smaller than 32
            'num_epochs': 120,
            'early_stopping_patience': 20
        },
        {
            'hidden_dims': [768, 384, 192],
            'dropout_rate': 0.55,
            'learning_rate': 0.00115,  # Scale LR with batch size
            'weight_decay': 1e-5,
            'batch_size': 64,
            'num_epochs': 120,
            'early_stopping_patience': 20
        },
        
        # ===== STRATEGIC COMBINED OPTIMIZATIONS =====
        # Fine-tuned dropout + LR
        {
            'hidden_dims': [768, 384, 192],
            'dropout_rate': 0.54,
            'learning_rate': 0.00105,
            'weight_decay': 1e-5,
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 20
        },
        # Fine-tuned dropout + weight decay
        {
            'hidden_dims': [768, 384, 192],
            'dropout_rate': 0.545,
            'learning_rate': 0.001,
            'weight_decay': 1.2e-5,
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 20
        },
        # Optimal dropout + adjusted batch + LR
        {
            'hidden_dims': [768, 384, 192],
            'dropout_rate': 0.55,
            'learning_rate': 0.00095,
            'weight_decay': 1e-5,
            'batch_size': 48,
            'num_epochs': 120,
            'early_stopping_patience': 20
        },
        # Deeper network + optimal dropout + more training
        {
            'hidden_dims': [768, 384, 192, 96],
            'dropout_rate': 0.54,
            'learning_rate': 0.001,
            'weight_decay': 1.2e-5,
            'batch_size': 32,
            'num_epochs': 130,
            'early_stopping_patience': 22
        },
        # Wider shallow + optimal settings
        {
            'hidden_dims': [800, 400, 200],
            'dropout_rate': 0.55,
            'learning_rate': 0.001,
            'weight_decay': 1e-5,
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 20
        },
        
        # ===== EXTENDED TRAINING & ALTERNATIVE APPROACHES =====
        # Best config with more epochs
        {
            'hidden_dims': [768, 384, 192],
            'dropout_rate': 0.55,
            'learning_rate': 0.001,
            'weight_decay': 1e-5,
            'batch_size': 32,
            'num_epochs': 150,
            'early_stopping_patience': 25
        },
        # Best config with more patience (less early stopping)
        {
            'hidden_dims': [768, 384, 192],
            'dropout_rate': 0.55,
            'learning_rate': 0.001,
            'weight_decay': 1e-5,
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 30
        },
        # Triple ensemble target: config 1
        {
            'hidden_dims': [768, 384, 192],
            'dropout_rate': 0.54,
            'learning_rate': 0.0009,
            'weight_decay': 1.2e-5,
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 20
        },
        # Triple ensemble target: config 2
        {
            'hidden_dims': [768, 384, 192],
            'dropout_rate': 0.55,
            'learning_rate': 0.001,
            'weight_decay': 1e-5,
            'batch_size': 48,
            'num_epochs': 120,
            'early_stopping_patience': 20
        },
        # Triple ensemble target: config 3
        {
            'hidden_dims': [768, 384, 192],
            'dropout_rate': 0.56,
            'learning_rate': 0.0011,
            'weight_decay': 7.5e-6,
            'batch_size': 24,
            'num_epochs': 120,
            'early_stopping_patience': 20
        },
    ]
    
    best_config = None
    best_score = 0
    results_log = []
    
    for i, config in enumerate(param_grid):
        print(f"\n{'='*60}")
        print(f"Testing configuration {i+1}/{len(param_grid)}")
        print(f"{'='*60}")
        print(config)
        
        # Split data
        train_split, val_split = train_test_split(
            train_data, test_size=0.2, random_state=42, 
            stratify=[d['label'] for d in train_data]
        )
        
        # Create datasets
        train_dataset = EmbeddingDataset(train_split, fit_scaler=True)
        val_dataset = EmbeddingDataset(val_split, scaler=train_dataset.scaler)
        
        train_loader = DataLoader(
            train_dataset, 
            batch_size=config['batch_size'], 
            shuffle=True
        )
        val_loader = DataLoader(
            val_dataset, 
            batch_size=config['batch_size'], 
            shuffle=False
        )
        
        # Train model
        _, f1, _, _ = train_model(train_loader, val_loader, device, config)
        
        # Log results
        results_log.append({
            'config_idx': i + 1,
            'f1_score': f1,
            'config': config
        })
        
        if f1 > best_score:
            best_score = f1
            best_config = config
            print(f"\n✓ New best configuration! F1: {f1:.4f}")
    
    # Print summary of all results
    print(f"\n{'='*60}")
    print("SUMMARY OF ALL CONFIGURATIONS:")
    print(f"{'='*60}")
    results_log.sort(key=lambda x: x['f1_score'], reverse=True)
    for i, result in enumerate(results_log[:5]):  # Top 5
        print(f"\n{i+1}. F1: {result['f1_score']:.4f}")
        print(f"   Config {result['config_idx']}: {result['config']}")
    
    print(f"\n{'='*60}")
    print("BEST CONFIGURATION:")
    print(best_config)
    print(f"Best F1 Score: {best_score:.4f}")
    print(f"{'='*60}")
    
    return best_config
# ============================================================================
# 5. CROSS-VALIDATION FOR ROBUST EVALUATION
# ============================================================================
def cross_validate_model(data, device, config, n_folds=5):
    """
    K-fold cross-validation for more robust performance estimate
    """
    labels = [d['label'] for d in data]
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
    
    fold_scores = []
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(data, labels)):
        print(f"\n{'='*60}")
        print(f"Fold {fold + 1}/{n_folds}")
        print(f"{'='*60}")
        
        train_data = [data[i] for i in train_idx]
        val_data = [data[i] for i in val_idx]
        
        # Create datasets
        train_dataset = EmbeddingDataset(train_data, fit_scaler=True)
        val_dataset = EmbeddingDataset(val_data, scaler=train_dataset.scaler)
        
        train_loader = DataLoader(
            train_dataset, 
            batch_size=config['batch_size'], 
            shuffle=True
        )
        val_loader = DataLoader(
            val_dataset, 
            batch_size=config['batch_size'], 
            shuffle=False
        )
        
        # Train model
        _, f1, _, _ = train_model(train_loader, val_loader, device, config)
        fold_scores.append(f1)
    
    print(f"\n{'='*60}")
    print("Cross-Validation Results:")
    print(f"Fold Scores: {[f'{s:.4f}' for s in fold_scores]}")
    print(f"Mean F1: {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}")
    print(f"{'='*60}")
    
    return fold_scores

# ============================================================================
# 6. MAIN TRAINING PIPELINE
# ============================================================================
def main():
    # Set random seeds for reproducibility
    torch.manual_seed(42)
    np.random.seed(42)
    
    # Device setup
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")
    
    # Load data
    print("\nLoading training data...")
    with open('train_part1.json', 'r') as f:
        data = json.load(f)
    
    print(f"Total samples: {len(data)}")
    labels = [d['label'] for d in data]
    print(f"Class distribution: 0={labels.count(0)}, 1={labels.count(1)}")
    
    # Option 2: Hyperparameter search (uncomment to use)
    best_config = hyperparameter_search(data, device)
    
    # Option 3: Cross-validation (uncomment to use)
    #cv_scores = cross_validate_model(data, device, best_config, n_folds=5)
    
    # Final training on full data with train/val split
    print("\nTraining final model...")
    train_data, val_data = train_test_split(
        data, test_size=0.2, random_state=42,
        stratify=[d['label'] for d in data]
    )
    
    train_dataset = EmbeddingDataset(train_data, fit_scaler=True)
    val_dataset = EmbeddingDataset(val_data, scaler=train_dataset.scaler)
    
    train_loader = DataLoader(
        train_dataset, 
        batch_size=best_config['batch_size'], 
        shuffle=True
    )
    val_loader = DataLoader(
        val_dataset, 
        batch_size=best_config['batch_size'], 
        shuffle=False
    )
    
    model, best_f1, train_losses, val_f1s = train_model(
        train_loader, val_loader, device, best_config
    )
    
    # Load best model for prediction
    checkpoint = torch.load('best_model.pth')
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    
    # Make predictions on test set
    print("\nMaking predictions on test set...")
    with open('test.json', 'r') as f:
        test_data = json.load(f)
    
    test_dataset = EmbeddingDataset(
        test_data, 
        scaler=train_dataset.scaler, 
        is_test=True
    )
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)
    
    predictions = []
    test_ids = []
    
    with torch.no_grad():
        for features, ids in test_loader:
            features = features.to(device)
            outputs = model(features)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            predictions.extend(preds)
            # Convert tensor IDs to regular integers/numbers
            if isinstance(ids, torch.Tensor):
                test_ids.extend(ids.cpu().numpy().tolist())
            else:
                test_ids.extend(ids)
    
    # Create submission
    submission = pd.DataFrame({
        'row_id': test_ids,
        'target': predictions
    })
    submission.to_csv('ann_v9.csv', index=False)
    print("\n✓ Submission file created: submission.csv")
    print(f"Predictions: 0={predictions.count(0)}, 1={predictions.count(1)}")

if __name__ == "__main__":
    main()

Using device: cuda

Loading training data...
Total samples: 1530
Class distribution: 0=1326, 1=204

Testing configuration 1/29
{'hidden_dims': [768, 384, 192], 'dropout_rate': 0.55, 'learning_rate': 0.001, 'weight_decay': 1e-05, 'batch_size': 32, 'num_epochs': 120, 'early_stopping_patience': 20}
Epoch 1/120
  Train Loss: 2.8490, Train F1: 0.5165
  Val Loss: 0.7369, Val F1: 0.6381
  LR: 0.001000
  ✓ New best F1: 0.6381
  ✓ New best F1: 0.7518
Epoch 5/120
  Train Loss: 1.5509, Train F1: 0.6330
  Val Loss: 0.8869, Val F1: 0.6373
  LR: 0.001000
Epoch 10/120
  Train Loss: 0.7614, Train F1: 0.7583
  Val Loss: 1.0248, Val F1: 0.6774
  LR: 0.000500
Epoch 15/120
  Train Loss: 0.6170, Train F1: 0.7939
  Val Loss: 1.0603, Val F1: 0.6415
  LR: 0.000250
Epoch 20/120
  Train Loss: 0.3423, Train F1: 0.8429
  Val Loss: 1.1012, Val F1: 0.6537
  LR: 0.000125
  → Learning rate reduced from 0.000250 to 0.000125

Early stopping triggered after 22 epochs

Training completed. Best Val F1: 0.7518

Final Valid

c:\Users\Lenovo\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Lenovo\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Lenovo\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(

Epoch 1/120
  Train Loss: 4.4514, Train F1: 0.4617
  Val Loss: 0.7673, Val F1: 0.6885
  LR: 0.000900
  ✓ New best F1: 0.6885
  ✓ New best F1: 0.7296
Epoch 5/120
  Train Loss: 1.7658, Train F1: 0.6217
  Val Loss: 1.0113, Val F1: 0.6342
  LR: 0.000900
Epoch 10/120
  Train Loss: 0.9301, Train F1: 0.7577
  Val Loss: 1.3577, Val F1: 0.6325
  LR: 0.000450
Epoch 15/120
  Train Loss: 0.6156, Train F1: 0.8168
  Val Loss: 1.4495, Val F1: 0.6564
  LR: 0.000225
  → Learning rate reduced from 0.000450 to 0.000225
Epoch 20/120
  Train Loss: 0.4910, Train F1: 0.8453
  Val Loss: 1.5468, Val F1: 0.6590
  LR: 0.000225

Early stopping triggered after 23 epochs

Training completed. Best Val F1: 0.7296

Final Validation Classification Report:
               precision    recall  f1-score   support

Not Important       0.90      0.95      0.92       265
    Important       0.46      0.29      0.36        41

     accuracy                           0.86       306
    macro avg       0.68      0.62      0.64  

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_6964\1413409710.py:693: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load('best_model.pth')



✓ Submission file created: submission.csv
Predictions: 0=392, 1=108
